# Week 5 Demonstration: Naive Bayes Text Classification

**Instructor demonstration notebook — fully worked example.**

Support Vector Machines (this week's main topic) are a *discriminative* classifier: they learn a boundary that separates the classes. **Naive Bayes** takes the opposite, *generative* approach — it models how each class *produces* its features and applies Bayes' rule to pick the most probable class. It is fast, needs little data, and is the classic baseline for **text classification**, which is why it shows up so often in information-science work.

This notebook walks through Naive Bayes end-to-end on a text dataset, then contrasts the different NB variants in scikit-learn.

> **Dataset: 20 Newsgroups.** ~18,000 public Usenet posts across 20 topics. We use four clearly-benign topics — `comp.graphics`, `rec.autos`, `rec.sport.baseball`, `sci.space` — and predict which newsgroup a post came from. The data ships with scikit-learn.

### The idea in one line

Bayes' rule: $P(\text{class} \mid \text{words}) \propto P(\text{class}) \times \prod_i P(\text{word}_i \mid \text{class})$.

The "naive" part is assuming the words are **conditionally independent** given the class. That assumption is obviously false for language, yet the classifier works remarkably well.

### API References

| Task | Documentation |
|------|---------------|
| Load text data | [fetch_20newsgroups](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html) |
| Bag-of-words counts | [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) |
| TF-IDF features | [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) |
| Multinomial NB | [MultinomialNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) |
| Bernoulli NB | [BernoulliNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.BernoulliNB.html) |
| Complement NB | [ComplementNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.ComplementNB.html) |
| Gaussian NB | [GaussianNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) |
| Pipelines | [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week5/week5_naive_bayes_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=11)

## Step 1: Load the Text Data

We fetch the *train* and *test* splits for four newsgroups. We strip the headers, signature footers, and quoted text with `remove=(...)` so the classifier has to learn from the actual message content rather than giveaway metadata (this makes the task honest — and harder).

In [ ]:
from sklearn.datasets import fetch_20newsgroups

categories = ['comp.graphics', 'rec.autos', 'rec.sport.baseball', 'sci.space']
remove = ('headers', 'footers', 'quotes')

train = fetch_20newsgroups(subset='train', categories=categories,
                           remove=remove, random_state=42)
test = fetch_20newsgroups(subset='test', categories=categories,
                          remove=remove, random_state=42)

print('Training documents:', len(train.data))
print('Test documents:    ', len(test.data))
print('Classes:', train.target_names)

In [ ]:
# Look at one example document and its label
print('LABEL:', train.target_names[train.target[10]])
print('-' * 60)
print(train.data[10][:500])

## Step 2: From Text to Numbers — Bag of Words

A model cannot consume raw strings. `CountVectorizer` turns each document into a vector of **word counts** over the vocabulary — the classic "bag of words" representation (word order is discarded, only counts remain). `stop_words='english'` drops common filler words like *the*, *is*, *and*.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')
X_train_counts = vectorizer.fit_transform(train.data)

print('Document-term matrix shape:', X_train_counts.shape)
print('  (rows = documents, columns = vocabulary words)')
print('Vocabulary size:', len(vectorizer.vocabulary_))
print('Example words:', list(vectorizer.vocabulary_)[:8])

## Step 3: Train a Multinomial Naive Bayes Classifier

`MultinomialNB` is the NB variant for **count data** — exactly what a bag of words produces. We bundle the vectorizer and the classifier into a single `Pipeline` so that the same text transformation is applied consistently to train and test.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

nb_clf = make_pipeline(CountVectorizer(stop_words='english'), MultinomialNB())
nb_clf.fit(train.data, train.target)

y_pred = nb_clf.predict(test.data)
print(f'Test accuracy (CountVectorizer + MultinomialNB): {accuracy_score(test.target, y_pred):.3f}')

About 89% accuracy on a 4-way problem where random guessing would get 25% — and this is *after* we stripped away the headers that would have made it easy. Naive Bayes trained in a fraction of a second.

## Step 4: TF-IDF — Weighting Words by Informativeness

Raw counts over-weight words that simply appear a lot. **TF-IDF** (term frequency × inverse document frequency) down-weights words that appear in *many* documents (they are less discriminating) and up-weights rare, topic-specific words. Swapping `CountVectorizer` for `TfidfVectorizer` is usually a small, free improvement.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_clf = make_pipeline(TfidfVectorizer(stop_words='english'), MultinomialNB())
tfidf_clf.fit(train.data, train.target)

y_pred_tfidf = tfidf_clf.predict(test.data)
print(f'Test accuracy (TfidfVectorizer + MultinomialNB): {accuracy_score(test.target, y_pred_tfidf):.3f}')

## Step 5: Where Does It Get Confused?

A confusion matrix shows which topics are mistaken for which. We expect the two vehicle/sport/tech topics to be cleanly separated, with any confusion happening between the most semantically similar pairs.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    test.target, y_pred_tfidf, display_labels=train.target_names,
    xticks_rotation=45, ax=ax, colorbar=False)
plt.title('Naive Bayes — Newsgroup Confusion Matrix')
plt.tight_layout()
plt.show()

print(classification_report(test.target, y_pred_tfidf,
                            target_names=train.target_names))

## Step 6: Classify Brand-New Sentences

The trained pipeline takes raw strings, so we can hand it sentences it has never seen and watch it assign a topic. This is the part students find convincing.

In [ ]:
new_docs = [
    'The pitcher threw a fastball and the batter hit a home run to win the game.',
    'NASA launched a rocket that will orbit the moon and study cosmic radiation.',
    'I need a faster GPU to render high-resolution 3D images and textures.',
    'The new sedan gets great mileage and the engine has plenty of horsepower.',
]

predicted = tfidf_clf.predict(new_docs)
for doc, label in zip(new_docs, predicted):
    print(f'[{train.target_names[label]:>20}]  {doc}')

## Step 7: The Naive Bayes Variants

scikit-learn ships several NB classifiers, each matching a different feature distribution:

- **`MultinomialNB`** — for **counts** (word frequencies). The default for text.
- **`ComplementNB`** — a variant of Multinomial designed to be more robust on **imbalanced** text; often edges out Multinomial.
- **`BernoulliNB`** — for **binary** features (word present / absent, ignoring how many times). Loses information when counts matter.
- **`GaussianNB`** — for **continuous** features assumed normally distributed. Wrong choice for word counts, but the right one for numeric tabular data.

Let's compare the three text-appropriate ones on the same TF-IDF features.

In [ ]:
from sklearn.naive_bayes import BernoulliNB, ComplementNB

for name, clf in [('MultinomialNB', MultinomialNB()),
                  ('ComplementNB', ComplementNB()),
                  ('BernoulliNB', BernoulliNB())]:
    model = make_pipeline(TfidfVectorizer(stop_words='english'), clf)
    model.fit(train.data, train.target)
    acc = accuracy_score(test.target, model.predict(test.data))
    print(f'{name:16s} test accuracy = {acc:.3f}')

`MultinomialNB` and `ComplementNB` land around 0.90, while `BernoulliNB` trails (~0.74) because collapsing counts to present/absent throws away useful signal.

## Step 8: The Smoothing Hyperparameter `alpha`

Naive Bayes multiplies per-word probabilities. If a word never appeared in a class during training, its probability would be **zero**, which would zero out the entire product. **Additive (Laplace/Lidstone) smoothing** adds a small `alpha` to every count so nothing is ever exactly zero. `alpha=1.0` is the default; tuning it can help.

In [ ]:
for alpha in [0.001, 0.01, 0.1, 1.0, 10.0]:
    model = make_pipeline(TfidfVectorizer(stop_words='english'),
                          MultinomialNB(alpha=alpha))
    model.fit(train.data, train.target)
    acc = accuracy_score(test.target, model.predict(test.data))
    print(f'alpha = {alpha:6.3f}   test accuracy = {acc:.3f}')

There is usually a gentle sweet spot (here around `alpha=0.1`). Too small and rare words get over-trusted; too large and the smoothing washes out real signal.

## Step 9: Peeking at What the Model Learned

Because Naive Bayes stores a log-probability for every word in every class, we can ask: *which words most strongly indicate each topic?* This interpretability is a genuine strength of the model.

In [ ]:
# Fit a bare vectorizer + classifier so we can reach inside
vec = TfidfVectorizer(stop_words='english')
Xtr = vec.fit_transform(train.data)
clf = MultinomialNB(alpha=0.1).fit(Xtr, train.target)

feature_names = np.array(vec.get_feature_names_out())
for i, topic in enumerate(train.target_names):
    top = np.argsort(clf.feature_log_prob_[i])[-10:][::-1]
    print(f'{topic:>20}: ' + ', '.join(feature_names[top]))

The top words are exactly what you'd hope: space/orbit/launch for `sci.space`, car/engine/dealer for `rec.autos`, and so on. That transparency — plus the speed and small-data appetite — is why Naive Bayes remains a go-to baseline before reaching for heavier models.

## Takeaways

- Naive Bayes is a **generative** classifier built on Bayes' rule plus a (false but useful) conditional-independence assumption.
- For **text**, represent documents as bag-of-words counts or TF-IDF, then use **`MultinomialNB`** (or `ComplementNB`).
- **`alpha`** smoothing prevents zero-probability blowups; tune it.
- It is fast, works with little data, and is interpretable — an excellent baseline to beat before trying SVMs or deep models.